### Display MTEB results
MTEB results are saved in .json files, one for each task. 
This notebook aggregates the results for multiple tasks and multiple models.

In [1]:
import os
os.getcwd()

'/Users/lena/Documents/GitHub/text_embedding/notebooks'

In [2]:
import mteb
import json
import pandas as pd

In [3]:
# get task type 
mteb.get_task("ArguAna").metadata.type

'Retrieval'

In [4]:
data = {}
main_dir = "../MTEB/sparse_results"

# Traverse folders and subfolders
for (root, dirs, files) in os.walk(main_dir):
    # Identify model version from the folder structure
    model_version = os.path.basename(root)
    print(model_version)
        
    for file in files:
        # Skip unwanted files
        if file in {"model_meta.json"} or not file.endswith('.json'):
            continue
            
        file_path = os.path.join(root, file)
        try:
            # Read JSON and extract necessary fields
            with open(file_path, 'r') as f:
                json_data = json.load(f)
                task_name = json_data.get("task_name")
                main_score = json_data.get("scores", {}).get("test", {})[0]["main_score"]
                main_score = round(main_score*100, 2)
                    
                if task_name and main_score is not None:
                    if task_name not in data:
                        data[task_name] = {}
                    data[task_name][model_version] = main_score
        except (json.JSONDecodeError, KeyError):
            print(f"Error parsing file: {file_path}")


sparse_results
sentence-transformers__average_word_embeddings_glove.6B.300d
no_revision_available
Tfidf
svd300_log
svd50_log
svd_log_piecewise
1.0
Error parsing file: ../MTEB/sparse_results/Tfidf/1.0/BibleNLPBitextMining.json
tfidf_log_run2
tfidf_rnd100_log
svd500_log
svd_log_run2
svd_log_old_run2
svd200_log
svd
tfidf_rnd768_log
tfidf_log_run1
svd_log_run1
svd_log_old_run1


In [5]:
df = pd.DataFrame.from_dict(data, orient='index')
df.index.name = "task_name"
df["task_types"] = [mteb.get_task(task).metadata.type for task in df.index]
df

,no_revision_available,1.0,svd,svd_log_run1,svd300_log,svd50_log,svd_log_piecewise,tfidf_log_run2,tfidf_rnd100_log,svd500_log,svd_log_run2,svd200_log,tfidf_rnd768_log,tfidf_log_run1,svd_log_old_run1,svd_log_old_run2,task_types
task_name,,,,,,,,,,,,,,,,,
AmazonPolarityClassification,63.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Classification
Banking77Classification,67.8,63.88,2.81,57.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Classification
MedrxivClusteringP2P,NaN,19.30,NaN,30.11,29.87,29.81,30.11,22.02,11.16,29.39,30.11,29.97,17.13,20.82,30.00,NaN,Clustering
ArxivClusteringP2P,NaN,26.72,NaN,40.23,40.60,39.75,40.23,34.79,8.32,40.84,40.23,40.36,29.12,35.28,41.73,NaN,Clustering
MindSmallReranking,NaN,23.44,NaN,26.63,27.31,25.75,NaN,22.49,24.75,NaN,26.63,27.04,27.00,23.45,NaN,NaN,Reranking
SciDocsRR,NaN,62.30,NaN,52.71,58.65,48.32,NaN,62.32,52.08,60.46,52.71,56.49,61.72,62.32,NaN,NaN,Reranking
ArguAna,NaN,42.48,NaN,41.35,51.14,32.53,NaN,52.52,11.63,54.25,41.35,48.24,43.38,52.54,NaN,NaN,Retrieval
STS15,NaN,74.01,NaN,52.51,61.68,46.04,NaN,73.42,70.74,66.10,52.51,57.74,73.06,73.43,NaN,NaN,STS
SCIDOCS,NaN,13.31,NaN,5.31,7.72,4.08,NaN,14.73,3.83,9.61,5.31,6.77,12.57,14.69,NaN,NaN,Retrieval


In [6]:
task_selection = ["ArguAna", "ArxivClusteringP2P", "BiorxivClusteringP2P", "MedrxivClusteringP2P", "MindSmallReranking",
                 "RedditClusteringP2P", "SCIDOCS", "SciDocsRR", "StackExchangeClusteringP2P", "STS15", "STS16",
                 "STSBenchmark"]

df.loc[task_selection].sort_values("task_types")

,no_revision_available,1.0,svd,svd_log_run1,svd300_log,svd50_log,svd_log_piecewise,tfidf_log_run2,tfidf_rnd100_log,svd500_log,svd_log_run2,svd200_log,tfidf_rnd768_log,tfidf_log_run1,svd_log_old_run1,svd_log_old_run2,task_types
task_name,,,,,,,,,,,,,,,,,
ArxivClusteringP2P,NaN,26.72,NaN,40.23,40.60,39.75,40.23,34.79,8.32,40.84,40.23,40.36,29.12,35.28,41.73,NaN,Clustering
BiorxivClusteringP2P,NaN,20.56,NaN,33.74,33.64,33.62,33.74,26.21,4.00,33.26,33.74,33.28,18.14,26.44,33.80,NaN,Clustering
MedrxivClusteringP2P,NaN,19.30,NaN,30.11,29.87,29.81,30.11,22.02,11.16,29.39,30.11,29.97,17.13,20.82,30.00,NaN,Clustering
RedditClusteringP2P,NaN,31.54,NaN,34.59,38.23,33.98,34.59,39.94,11.37,40.91,34.59,36.51,32.42,40.34,45.96,46.5,Clustering
StackExchangeClusteringP2P,NaN,18.80,NaN,34.12,31.48,36.00,34.11,17.40,17.87,29.73,34.12,32.62,17.00,17.53,34.01,NaN,Clustering
MindSmallReranking,NaN,23.44,NaN,26.63,27.31,25.75,NaN,22.49,24.75,NaN,26.63,27.04,27.00,23.45,NaN,NaN,Reranking
SciDocsRR,NaN,62.30,NaN,52.71,58.65,48.32,NaN,62.32,52.08,60.46,52.71,56.49,61.72,62.32,NaN,NaN,Reranking
ArguAna,NaN,42.48,NaN,41.35,51.14,32.53,NaN,52.52,11.63,54.25,41.35,48.24,43.38,52.54,NaN,NaN,Retrieval
SCIDOCS,NaN,13.31,NaN,5.31,7.72,4.08,NaN,14.73,3.83,9.61,5.31,6.77,12.57,14.69,NaN,NaN,Retrieval


In [11]:
df[["svd50_log", "svd_log_run2", "svd200_log", "svd300_log", "svd500_log", "task_types"]].loc[task_selection].sort_values("task_types")

,svd50_log,svd_log_run2,svd200_log,svd300_log,svd500_log,task_types
task_name,,,,,,
ArxivClusteringP2P,39.75,40.23,40.36,40.60,40.84,Clustering
BiorxivClusteringP2P,33.62,33.74,33.28,33.64,33.26,Clustering
MedrxivClusteringP2P,29.81,30.11,29.97,29.87,29.39,Clustering
RedditClusteringP2P,33.98,34.59,36.51,38.23,40.91,Clustering
StackExchangeClusteringP2P,36.00,34.12,32.62,31.48,29.73,Clustering
MindSmallReranking,25.75,26.63,27.04,27.31,NaN,Reranking
SciDocsRR,48.32,52.71,56.49,58.65,60.46,Reranking
ArguAna,32.53,41.35,48.24,51.14,54.25,Retrieval
SCIDOCS,4.08,5.31,6.77,7.72,9.61,Retrieval


In [14]:
df[["tfidf_log_run2", "svd_log_old_run1", "svd_log_run2", "svd_log_piecewise", "task_types"]].loc[task_selection].sort_values("task_types")

,tfidf_log_run2,svd_log_old_run1,svd_log_run2,svd_log_piecewise,task_types
task_name,,,,,
ArxivClusteringP2P,34.79,41.73,40.23,40.23,Clustering
BiorxivClusteringP2P,26.21,33.80,33.74,33.74,Clustering
MedrxivClusteringP2P,22.02,30.00,30.11,30.11,Clustering
RedditClusteringP2P,39.94,45.96,34.59,34.59,Clustering
StackExchangeClusteringP2P,17.40,34.01,34.12,34.11,Clustering
MindSmallReranking,22.49,NaN,26.63,NaN,Reranking
SciDocsRR,62.32,NaN,52.71,NaN,Reranking
ArguAna,52.52,NaN,41.35,NaN,Retrieval
SCIDOCS,14.73,NaN,5.31,NaN,Retrieval
